In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from finder_core.settings import load_settings
import json
import time

import numpy as np
import pandas as pd
import yfinance as yf

from IPython.display import display

parameters = json.loads(Path('config/settings.json').read_text())
ticker = parameters['ticker']
settings = load_settings(root, ticker=ticker)
output_root = root
analysis_start_date = settings['analysis_start_date']
end_date = settings['analysis_end_date']
window_size = int(settings['window_size'])
download_start_date = '2006-01-01'
max_interpolation_days = 3
market_registry = settings['market_registry']['markets']
refinitiv_assets = {key: value['price_sources']['lseg']['symbol'] for (key, value) in market_registry.items() if value['price_sources']['lseg']['symbol']}
yahoo_assets = {key: value['price_sources']['yahoo']['symbol'] if value['price_sources']['yahoo']['allow_fill'] else None for (key, value) in market_registry.items()}
other_source_files = {}
refinitiv_fields = {'TRDPRC_1': 'Close', 'HIGH_1': 'High', 'LOW_1': 'Low', 'OPEN_PRC': 'Open', 'ACVOL_UNS': 'Volume'}
columns = list(refinitiv_fields.values())
print(f'Analysis: {analysis_start_date} to {end_date} | Download including warm-up: {download_start_date}')

In [ ]:
update_market_data = False

In [ ]:
def download_refinitiv_market(ric):
    for attempt in range(3):
        try:
            response = historical_pricing.summaries.Definition(universe=ric, interval=historical_pricing.Intervals.DAILY, start=download_start_date, end=end_date, fields=list(refinitiv_fields), adjustments='unadjusted').get_data()
            break
        except Exception as exc:
            transient = any((value in str(exc).lower() for value in ('timeout', '503', '502', 'failed to connect')))
            if not transient or attempt == 2:
                raise
            time.sleep(2 ** attempt)
    raw = response.data.raw
    while isinstance(raw, list) and len(raw) == 1:
        raw = raw[0]
    if raw.get('universe', {}).get('ric') != ric:
        raise ValueError(f'Returned RIC does not match {ric}')
    frame = pd.DataFrame(raw.get('data', []), columns=[item['name'] for item in raw.get('headers', [])])
    if frame.empty or not {'DATE', 'TRDPRC_1'}.issubset(frame.columns):
        raise ValueError(f'No dated closing prices returned for {ric}')
    frame['DATE'] = pd.to_datetime(frame['DATE'], errors='raise', utc=True).dt.tz_convert(None)
    frame = frame.set_index('DATE').sort_index()
    if frame.index.has_duplicates:
        raise ValueError(f'Duplicate dates returned for {ric}')
    frame = frame.loc[download_start_date:end_date].rename(columns=refinitiv_fields)
    frame = frame.reindex(columns=list(refinitiv_fields.values())).apply(pd.to_numeric, errors='coerce')
    if frame['Close'].dropna().le(0).any():
        raise ValueError(f'Nonpositive closing price returned for {ric}')
    frame.index.name = 'Date'
    return (frame, raw)

def normalize_prices(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    frame.index = pd.to_datetime(frame.index).tz_localize(None).normalize()
    if frame.index.has_duplicates:
        raise ValueError('Duplicate market dates.')
    frame = frame.reindex(columns=columns).apply(pd.to_numeric, errors='coerce').astype(float).sort_index()
    frame = frame.loc[download_start_date:end_date].replace([np.inf, -np.inf], np.nan)
    frame.index.name = 'Date'
    return frame

def fill_prices(primary, fallback, fallback_name, primary_name='LSEG'):
    dates = primary.index.union(fallback.index).sort_values().rename('Date')
    frame = primary.reindex(dates).copy()
    source = pd.DataFrame('', index=dates, columns=columns).mask(frame.notna(), primary_name)
    fallback = fallback.reindex(dates)
    filled = frame.isna() & fallback.notna()
    frame = frame.fillna(fallback).astype(float)
    source = source.mask(filled, fallback_name)
    return (frame, source)

def interpolate_small_gaps(frame, source):
    (frame, source) = (frame.copy(), source.copy())
    for column in columns:
        missing = frame[column].isna()
        gap_size = missing.groupby(missing.ne(missing.shift()).cumsum()).transform('sum')
        interpolated = frame[column].interpolate(method='linear', limit_area='inside')
        filled = missing & gap_size.le(max_interpolation_days) & interpolated.notna()
        frame.loc[filled, column] = interpolated.loc[filled]
        source.loc[filled, column] = 'interpolation'
    return (frame, source)

if update_market_data:
    import lseg.data as ld
    from lseg.data.content import historical_pricing
    refinitiv_data = {}
    try:
        session = ld.open_session(name='desktop.workspace', app_key=os.getenv('LSEG_APP_KEY'))
        if session.open_state.name != 'Opened':
            raise ConnectionError('Open and sign in to LSEG Workspace on this computer.')
        for (asset, ric) in refinitiv_assets.items():
            (frame, raw) = download_refinitiv_market(ric)
            folder = (output_root / 'Data') / asset / 'Refinitiv'
            folder.mkdir(parents=True, exist_ok=True)
            (folder / 'historical_pricing.json').write_text(json.dumps(raw, ensure_ascii=False, indent=2), encoding='utf-8')
            undated_empty_rows = 0
            duplicate_date_rows = 0
            if asset == 'CBU0.L':
                eod = ld.get_data(ric, ['TR.PriceClose.date', 'TR.PriceClose', 'TR.PriceOpen', 'TR.PriceHigh', 'TR.PriceLow', 'TR.Volume'], parameters={'SDate': download_start_date, 'EDate': end_date, 'Frq': 'D', 'Adjusted': 0})
                eod.to_csv(folder / 'eod_prices.csv', index=False)
                undated = eod['Date'].isna()
                if (undated & (eod[['Price Close', 'Price Open', 'Price High', 'Price Low']].notna().any(axis=1) | eod['Volume'].fillna(0).ne(0))).any():
                    raise ValueError('CBU0.L: undated rows contain market values.')
                undated_empty_rows = int(undated.sum())
                eod = eod.loc[~undated].copy()
                if undated_empty_rows:
                    print(f'CBU0.L: excluded {undated_empty_rows} undated empty rows; original response retained.')
                if eod.empty:
                    raise ValueError('CBU0.L: no dated exchange closing prices returned.')
                duplicate_date_rows = int(eod['Date'].duplicated().sum())
                if duplicate_date_rows:
                    if eod.groupby('Date').nunique(dropna=True).gt(1).any().any():
                        raise ValueError('CBU0.L: conflicting values on duplicate dates.')
                    eod = eod.groupby('Date', as_index=False).first()
                    print(f'CBU0.L: merged {duplicate_date_rows} duplicate-date rows with no conflicting values.')
                frame = normalize_prices(eod.set_index('Date').rename(columns={'Price Close': 'Close', 'Price Open': 'Open', 'Price High': 'High', 'Price Low': 'Low'}))
            frame.to_csv(folder / 'df_OHLCV.csv')
            metadata = {'asset': asset, 'ric': ric, 'requested_start': download_start_date, 'requested_end': end_date, 'rows': len(frame), 'start': str(frame.index.min().date()), 'end': str(frame.index.max().date()), 'missing_values': frame.isna().sum().to_dict(), 'adjustments': 'unadjusted', 'price_source': 'LSEG exchange EOD (TR.Price fields)' if asset == 'CBU0.L' else 'LSEG Historical Pricing', 'retrieved_at_utc': pd.Timestamp.now(tz='UTC').isoformat()}
            metadata['undated_empty_rows_excluded'] = undated_empty_rows
            metadata['duplicate_date_rows_merged'] = duplicate_date_rows
            metadata['market_definition'] = market_registry[asset]
            (folder / 'metadata.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
            refinitiv_data[asset] = normalize_prices(frame)
            print(f'{asset}: {len(frame)} LSEG rows, {int(frame.isna().sum().sum())} missing values')
    finally:
        ld.close_session()
    yahoo_data = {}
    yahoo_errors = {}
    for (asset, symbol) in yahoo_assets.items():
        folder = (output_root / 'Data') / asset / 'yfinance'
        folder.mkdir(parents=True, exist_ok=True)
        yahoo_data[asset] = pd.DataFrame(columns=columns, index=pd.DatetimeIndex([], name='Date'))
        if symbol is None:
            yahoo_errors[asset] = 'No verified Yahoo equivalent configured.'
            continue
        try:
            raw = yf.download(symbol, start=download_start_date, end=(pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime('%Y-%m-%d'), auto_adjust=False, progress=False)
            if raw.empty:
                raise ValueError('Yahoo returned no data.')
            raw.to_csv(folder / 'download.csv')
            yahoo_data[asset] = normalize_prices(raw).dropna(how='all')
            yahoo_data[asset].to_csv(folder / 'df_OHLCV.csv')
        except Exception as exc:
            yahoo_errors[asset] = f'{type(exc).__name__}: {exc}'
            print(f'{asset}: {yahoo_errors[asset]}')
    market_data = {}
    fill_sources = {}
    summary = []
    for (asset, primary) in refinitiv_data.items():
        source_config = market_registry[asset]['price_sources']
        source_frames = {'lseg': primary, 'yahoo': yahoo_data[asset]}
        source_names = {'lseg': 'LSEG', 'yahoo': 'yfinance'}
        source_order = sorted((key for key in source_frames if source_config[key]['symbol'] and source_config[key]['allow_fill']), key=lambda key: source_config[key]['priority'])
        if not source_order:
            raise ValueError(f'{asset}: no enabled price source.')
        first = source_order[0]
        second = source_order[1] if len(source_order) > 1 else None
        fallback = source_frames[second] if second else primary.iloc[:0]
        (frame, source) = fill_prices(source_frames[first], fallback, source_names[second] if second else '', source_names[first])
        if asset in other_source_files and frame.isna().any().any():
            other = normalize_prices(pd.read_csv(other_source_files[asset], index_col='Date', parse_dates=['Date']))
            other = other.dropna(how='all').reindex(frame.index)
            filled = frame.isna() & other.notna()
            frame = frame.fillna(other)
            source = source.mask(filled, str(other_source_files[asset]))
        (frame, source) = interpolate_small_gaps(frame, source)
        if frame[columns[:-1]].le(0).any().any() or frame['Volume'].lt(0).any():
            raise ValueError(f'{asset}: nonpositive prices or negative volume.')
        frame['return'] = np.log(frame['Close']).diff()
        (market_data[asset], fill_sources[asset]) = (frame, source)
        missing = int(frame[columns].isna().sum().sum())
        summary.append({'asset': asset, 'primary_source': source_names[first], 'start': frame.index.min().date(), 'end': frame.index.max().date(), 'rows': len(frame), 'fallback_cells': int(((source != '') & (source != source_names[first]) & (source != 'interpolation')).sum().sum()), 'interpolated_cells': int(source.eq('interpolation').sum().sum()), 'missing_values': missing, 'status': 'complete' if missing == 0 else 'incomplete', 'yahoo_note': yahoo_errors.get(asset, '')})
    display(pd.DataFrame(summary).set_index('asset'))

if update_market_data:
    for asset, frame in market_data.items():
        destination = output_root / 'Data' / asset / 'Market'
        destination.mkdir(parents=True, exist_ok=True)
        frame.iloc[1:].to_csv(destination / 'df_OHLCV.csv', index_label='Date')